# Notebook 1: The UTTT Game Engine

*Companion to the essay series on AlphaZero for Ultimate Tic-Tac-Toe.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thltsui/UlltimateTicTacToe/blob/Substack/substack/notebook_1_uttt_engine.ipynb)

---

This notebook is a guided tour of `01_game/` — the production game engine the rest of the codebase builds on. We import directly from it; nothing is re-implemented here.

By the end you will understand:
- How `GameState` represents the board as a `(9, 9)` array indexed by `(sub_board_idx, cell_idx)`
- How `apply_move()` enforces the "send your opponent" rule and returns an immutable new state
- How `encode_state()` turns a position into a 7-channel `(7, 9, 9)` tensor
- What a tournament of random games looks like in practice


## Setup

In [ ]:
# ── If running in Colab, uncomment: ──────────────────────────────────
# !git clone https://github.com/thltsui/UlltimateTicTacToe.git
# import os; os.chdir('UlltimateTicTacToe')
# ─────────────────────────────────────────────────────────────────────

import sys, os
from importlib import import_module
from collections import Counter
import numpy as np
import torch
import random

# Add repo root so that import_module('01_game.xxx') resolves correctly.
# When running from substack/ we need to go one level up.
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import using importlib (required because '01_game' starts with a digit)
board_mod      = import_module('01_game.board')
rules_mod      = import_module('01_game.rules')
visualizer_mod = import_module('01_game.visualizer')

GameState           = board_mod.GameState
create_initial_state = board_mod.create_initial_state
encode_state        = board_mod.encode_state
encode_move         = board_mod.encode_move
decode_move         = board_mod.decode_move

apply_move              = rules_mod.apply_move
get_legal_moves         = rules_mod.get_legal_moves
get_legal_move_mask     = rules_mod.get_legal_move_mask
check_sub_board_winner  = rules_mod.check_sub_board_winner
check_meta_winner       = rules_mod.check_meta_winner

render_board_ascii = visualizer_mod.render_board_ascii

print("Imports OK")


## 1. Board Representation: `GameState`

The entire game state lives in a single dataclass:

```python
@dataclass
class GameState:
    cells: np.ndarray           # shape (9, 9) — (sub_board_idx, cell_idx)
    sub_board_results: np.ndarray  # shape (9,) — result of each local board
    active_sub_board: int       # 0–8, or -1 = "free choice"
    current_player: int         # +1 or -1
    move_count: int
    is_terminal: bool
    winner: int | None          # 1, -1, 0 (draw), or None (ongoing)
```

**Two-level indexing.** Every cell is `cells[sub_board_idx, cell_idx]`, both 0–8.

```
Sub-board layout:      Cell layout within one sub-board:
  0 | 1 | 2                0 | 1 | 2
  ---------                ---------
  3 | 4 | 5                3 | 4 | 5
  ---------                ---------
  6 | 7 | 8                6 | 7 | 8
```

So `cells[4, 0]` = top-left cell of the centre sub-board.

**Player convention.** `+1` = X (player 1), `-1` = O (player 2). This symmetry matters for encoding: "my pieces" are always `cells == current_player`, opponent's are `cells == -current_player`, regardless of which player you are.

**Sub-board results.** `0` = ongoing, `1` = player 1 won, `-1` = player 2 won, `2` = draw.

**`active_sub_board = -1`** means free choice — used on the first move and whenever a player is sent to an already-decided board.


In [ ]:
state = create_initial_state()
print("cells shape:        ", state.cells.shape)
print("sub_board_results:  ", state.sub_board_results)
print("active_sub_board:   ", state.active_sub_board, "  (-1 = free choice)")
print("current_player:     ", state.current_player, "     (+1 = X)")
print("move_count:         ", state.move_count)
print("is_terminal:        ", state.is_terminal)
print("winner:             ", state.winner)
print()
print(render_board_ascii(state))


## 2. Move Encoding

Moves are **flat integers 0–80**:

```python
move_idx = sub_board_idx * 9 + cell_idx
```

`encode_move(sb, cell)` and `decode_move(move_idx)` convert between representations.
This flat index is exactly what the neural network's policy head outputs — a distribution over 81 positions.


In [ ]:
move = encode_move(4, 0)           # top-left cell of centre sub-board
print(f"encode_move(4, 0)  =  {move}")

sb, cell = decode_move(move)
print(f"decode_move({move})     =  sub_board={sb}, cell_idx={cell}")

# Roundtrip check for all 81 moves
for idx in range(81):
    sb2, c2 = decode_move(idx)
    assert encode_move(sb2, c2) == idx
print("Roundtrip encode↔decode: PASSED for all 81 moves")


## 3. Legal Moves and the "Send Your Opponent" Rule

`get_legal_moves(state)` returns a list of flat move indices.

**The key rule:** after playing in `cell_idx`, the **next player must play in sub-board `cell_idx`**. The cell's position within its local board becomes the opponent's destination. If that sub-board is already decided, the opponent gets free choice (`active_sub_board = -1`).


In [ ]:
state = create_initial_state()
print(f"Initial legal moves: {len(get_legal_moves(state))}  (free choice = all 81)")

# Play sb=4, cell=2 → opponent is sent to sub-board 2
state2 = apply_move(state, encode_move(4, 2))
legal = get_legal_moves(state2)

print(f"\nAfter encode_move(sb=4, cell=2):")
print(f"  active_sub_board = {state2.active_sub_board}  (sent to sub-board 2)")
print(f"  legal moves: {len(legal)}  (9 cells in sub-board 2)")
print(f"  all in sub-board 2? {all(decode_move(m)[0] == 2 for m in legal)}")

print()
print(render_board_ascii(state2))


## 4. Immutable State Transitions

`apply_move()` **never mutates its input** — it deep-copies the state and returns a new `GameState`. This is essential for MCTS, which holds many game states in memory simultaneously.


In [ ]:
state = create_initial_state()
state2 = apply_move(state, encode_move(4, 4))

assert state.cells[4, 4]  == 0,  "Original was mutated!"
assert state2.cells[4, 4] == 1,  "New state should have piece placed"
assert state2.current_player == -1
assert state2.active_sub_board  == 4   # cell_idx=4 → sent to sub-board 4

print("Immutability check: PASSED")
print(f"  Original player:  {state.current_player}")
print(f"  New state player: {state2.current_player}  (flipped to -1 = O)")
print(f"  New active_sub_board: {state2.active_sub_board}")


## 5. Win Detection

Two levels:
- **`check_sub_board_winner(cells_flat)`** — checks a flat length-9 array
- **`check_meta_winner(sub_board_results)`** — checks the 9 sub-board results as a meta-board

Both return `1` (player 1), `-1` (player 2), `2` (draw), or `0` (ongoing).

Drawn sub-boards (`2`) **don't count** for either player on the meta-board.


In [ ]:
# Local board tests
print("Row win (+1):  ", check_sub_board_winner(np.array([1,1,1,-1,-1,0,0,0,0],  dtype=np.int8)))
print("Draw  ( 2):    ", check_sub_board_winner(np.array([1,-1,1,1,-1,-1,-1,1,1], dtype=np.int8)))
print("Ongoing (0):   ", check_sub_board_winner(np.array([1,0,0,0,-1,0,0,0,0],   dtype=np.int8)))
print()

# Meta-board tests
meta_win = np.array([1, 1, 1, 0, -1, 0, -1, 0, 0], dtype=np.int8)
print("Meta win (+1): ", check_meta_winner(meta_win))

meta_draw_block = np.array([1, 2, -1, 2, 1, -1, -1, -1, 2], dtype=np.int8)
print("Meta ongoing:  ", check_meta_winner(meta_draw_block), "  (draws break lines)")


## 6. Neural Network Encoding: 7-Channel Tensor

`encode_state(state)` returns a `torch.Tensor` of shape `(7, 9, 9)`.

Always encoded **from the current player's perspective**:

| Channel | Content |
|---------|---------|
| 0 | Current player's pieces |
| 1 | Opponent's pieces |
| 2 | Active sub-board mask (legal cells) |
| 3 | Sub-boards won by current player |
| 4 | Sub-boards won by opponent |
| 5 | Drawn sub-boards |
| 6 | Turn indicator (0 if player +1 to move, 1 if player -1) |

**Spatial mapping** from `(sub_board_idx, cell_idx)` to `(row, col)` in 9×9:
```python
row = (sub_board_idx // 3) * 3 + (cell_idx // 3)
col = (sub_board_idx % 3)  * 3 + (cell_idx % 3)
```
Each sub-board occupies a contiguous 3×3 block — the natural input for convolutional layers.


In [ ]:
state = create_initial_state()
tensor = encode_state(state)
print(f"Tensor shape: {tuple(tensor.shape)}")
print()

names = [
    "Ch 0  current player pieces",
    "Ch 1  opponent pieces",
    "Ch 2  active sub-board mask",
    "Ch 3  sub-boards won by current player",
    "Ch 4  sub-boards won by opponent",
    "Ch 5  drawn sub-boards",
    "Ch 6  turn indicator",
]
for i, name in enumerate(names):
    print(f"  {name}: sum={tensor[i].sum().item():.0f}")


In [ ]:
# After one move: see the active sub-board light up
state = apply_move(create_initial_state(), encode_move(4, 2))
tensor = encode_state(state)  # current player is now -1 (O)

print(f"active_sub_board = {state.active_sub_board}, current_player = {state.current_player}")
print()
print("Ch 2 (active sub-board mask) — only sub-board 2 lit up:")
print(tensor[2].numpy().astype(int))
print()
print("Ch 6 (turn indicator = 1 because current player is -1):")
print(f"  sum = {tensor[6].sum().item():.0f}  (81 = all cells are 1.0)")


## 7. Legal Move Mask

`get_legal_move_mask(state)` returns a `(81,)` tensor used to **zero out illegal moves** in the network's policy output before softmax. This ensures the agent never samples an illegal action.


In [ ]:
state = create_initial_state()
mask = get_legal_move_mask(state)
print(f"Initial mask — legal moves: {mask.sum().int().item()} / 81")

state2 = apply_move(state, encode_move(4, 2))
mask2 = get_legal_move_mask(state2)
print(f"After encode_move(4,2)  — legal moves: {mask2.sum().int().item()} / 81")
print(f"  All in sub-board 2? {all(decode_move(i)[0]==2 for i in range(81) if mask2[i]==1)}")


## 8. Random Agent and Game Runner

Our baseline: pick uniformly from legal moves. Agent signature: `(GameState) -> int`.
This is the same interface MCTS will use.


In [ ]:
def random_agent(state: GameState) -> int:
    """Pick a uniformly random legal flat move index."""
    return random.choice(get_legal_moves(state))


def play_game(agent1=None, agent2=None, verbose=False):
    """
    Play a complete game. agent1=+1 (X), agent2=-1 (O).
    Returns (winner, move_count).
    winner: 1, -1, or 0 (draw).
    """
    if agent1 is None: agent1 = random_agent
    if agent2 is None: agent2 = random_agent
    state = create_initial_state()
    agents = {1: agent1, -1: agent2}
    while not state.is_terminal:
        state = apply_move(state, agents[state.current_player](state))
        if verbose: print(render_board_ascii(state))
    return state.winner, state.move_count


winner, moves = play_game()
outcome = {1: "X wins", -1: "O wins", 0: "Draw"}[winner]
print(f"Result: {outcome}  |  Moves played: {moves}")


## 9. Tournament

5,000 random-vs-random games. Expect X (+1) to win slightly more (first-mover advantage), with a substantial draw rate due to the game's complexity.


In [ ]:
def tournament(n=5000):
    results = Counter()
    lengths = []
    for _ in range(n):
        w, m = play_game()
        results[w] += 1
        lengths.append(m)
    arr = np.array(lengths)
    print(f"Tournament: {n:,} games")
    print(f"  X wins  (+1): {results[1]:5,}  ({100*results[1]/n:.1f}%)")
    print(f"  O wins  (-1): {results[-1]:5,}  ({100*results[-1]/n:.1f}%)")
    print(f"  Draws   ( 0): {results[0]:5,}  ({100*results[0]/n:.1f}%)")
    print(f"  Length — mean {arr.mean():.1f}  median {np.median(arr):.0f}  "
          f"min {arr.min()}  max {arr.max()}")

tournament()


---

## Summary

| Symbol | Meaning |
|--------|---------|
| `+1` | Player 1 (X) |
| `-1` | Player 2 (O) |
| `active_sub_board == -1` | Free choice |
| `sub_board_results[sb] == 2` | Sub-board is a draw |
| `winner == 0` | Game is a draw |
| move index | `sub_board_idx * 9 + cell_idx` |

*Next: Notebook 2 — Monte Carlo Tree Search using `apply_move()` and `get_legal_moves()` as the only game interface.*
